In [ ]:
import numpy as np
import torch
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from data.utils import get_datasets, precompute_features
from data.dataset import CachedDataset
from data.transforms import FeatureConfig
from models.train import train, predict
from models.evaluate import evaluate
from models.cnn import CNN
from models.transformer import Transformer
from models.cnn_transformer import CNNTransformer
from models.logistic_regression import LogisticRegressionBaseline
from models.utils import set_seed
from models.evaluate import evaluate
from plots import plot_f1_training_curves, build_summary_df, plot_summary_table, plot_metrics_comparison, plot_f1_comparison, plot_confusion_matrix

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)

In [ ]:
DATA_DIR   = Path('../data')
TRAIN_PATH = str(DATA_DIR / 'train')
VALID_PATH = str(DATA_DIR / 'valid')
TEST_PATH  = str(DATA_DIR / 'test')

SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH_SIZE = 64
LR = 3e-4

REPRESENTATIONS = ['mfcc', 'mel']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### Data Representation - MFCC vs Mel Spectrogram


- **MFCC** (`mfcc` format): 40 coefficients, normalized
- **Mel Spectrogram** (`mel` format): 64 mel bins, normalized


In [ ]:
def run_representation_experiment(model_factory, representations, seeds, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=device, model_name='Model'):
    """
    Train `model_factory(repr)` for every representation x seed combination.

    Args:
        model_factory: callable(repr_str) -> nn.Module  (already moved to device)
        representations: list of str, e.g. ['mfcc', 'mel']
        seeds: list of int
        ...

    Returns:
        results dict with structure:
            results[repr][seed] = {
                'history': training history dict,
                'valid_acc': float,
                'valid_loss': float,
                'valid_f1': float,
                'test_acc': float,
                'test_loss': float,
                'test_macro_f1': float,
                'test_weighted_f1': float,
                'test_cm': np.array,
            }
    """
    
    results = {}

    for repr_name in representations:
        results[repr_name] = {}

        train_ds, valid_ds, test_ds = get_datasets(data_format=repr_name, train_path=TRAIN_PATH, valid_path=VALID_PATH, test_path=TEST_PATH)
        precompute_features(train_ds, Path('/kaggle/working/cache/train'))
        precompute_features(valid_ds, Path('/kaggle/working/cache/valid'))
        precompute_features(test_ds, Path('/kaggle/working/cache/test'))
        train_ds_cached = CachedDataset('/kaggle/working/cache/train')
        valid_ds_cached = CachedDataset('/kaggle/working/cache/valid')
        test_ds_cached = CachedDataset('/kaggle/working/cache/test')

        for seed in seeds:
            print(f'\n[{model_name}] repr={repr_name} | seed={seed}')
            print('-' * 55)
            set_seed(seed)

            model = model_factory(repr_name).to(device)
            model, history = train(model, train_ds_cached, valid_ds_cached, epochs=epochs, batch_size=batch_size, lr=lr,
                device=str(device), verbose=True, verbose_interval=5)

            valid_acc = history['valid_acc'][-1]
            valid_loss = history['valid_loss'][-1]

            preds, labels = predict(model, test_ds_cached, device=str(device), batch_size=batch_size)
            test_results = evaluate(preds, labels, print_report=False)
            test_acc = test_results['acc']

            results[repr_name][seed] = {
                'history': history,
                'valid_acc': valid_acc,
                'valid_loss': valid_loss,
                'valid_f1': history['valid_f1'][-1],
                'test_acc': test_results['acc'],
                'test_macro_f1': test_results['macro_f1'],
                'test_weighted_f1': test_results['weighted_f1'],
                'test_cm': test_results['cm'],
            }

    return results

### CNN

In [ ]:
def cnn_factory(repr_name):
    """Return a CNN configured for the given representation."""
    # mfcc -> [1, 40, T]  |  mel -> [1, 64, T]
    return CNN(num_classes=12, in_channels=1, base_channels=32, dropout=0.3)

cnn_results = run_representation_experiment(model_factory=cnn_factory, representations=REPRESENTATIONS, seeds=SEEDS, model_name='CNN')

In [ ]:
plot_f1_training_curves(cnn_results, title_prefix='CNN')

In [ ]:
summary_cnn = build_summary_df({
    'CNN': cnn_results
})

In [ ]:
plot_summary_table(summary_cnn)

In [ ]:
plot_metrics_comparison(summary_cnn, title='CNN — Metrics Comparison')

In [ ]:
plot_f1_comparison(cnn_results, param_name='Representation', param_name_short='repr')

In [ ]:
best_repr = max(cnn_results, key=lambda r: np.mean([v['test_macro_f1'] for v in cnn_results[r].values()]))
best_seed = max(cnn_results[best_repr], key=lambda s: cnn_results[best_repr][s]['test_macro_f1'])

best_result = cnn_results[best_repr][best_seed]
plot_confusion_matrix(best_result['test_cm'])


### Transformer

In [ ]:
def transformer_factory(repr_name):
    n_features = 40 if repr_name == 'mfcc' else 64
    return Transformer(n_features=n_features, n_timesteps=101, num_classes=12, d_model=128, nhead=4, num_layers=4, dropout=0.1, pooling='mean')

transformer_results = run_representation_experiment(model_factory=transformer_factory, representations=REPRESENTATIONS, seeds=SEEDS, model_name='Transformer')

In [ ]:
plot_f1_training_curves(transformer_results, title_prefix='Transformer')

In [ ]:
summary_transformer = build_summary_df({
    'Transformer': transformer_results
})

In [ ]:
plot_summary_table(summary_transformer)

In [ ]:
plot_metrics_comparison(summary_transformer, title='Transformer — Metrics Comparison')

In [ ]:
plot_f1_comparison(transformer_results, param_name='Representation', param_name_short='repr')

In [ ]:
best_repr = max(transformer_results, key=lambda r: np.mean([v['test_macro_f1'] for v in transformer_results[r].values()]))
best_seed = max(transformer_results[best_repr], key=lambda s: transformer_results[best_repr][s]['test_macro_f1'])

best_result = transformer_results[best_repr][best_seed]
plot_confusion_matrix(best_result['test_cm'])

### CNN + Transformer

In [ ]:
def cnn_transformer_factory(repr_name):
    n_features = 40 if repr_name == 'mfcc' else 64
    return CNNTransformer(n_features=n_features, n_timesteps=101, num_classes=12, base_channels=32, d_model=128, nhead=4, num_layers=4, dropout=0.1, pooling='mean')

cnn_transformer_results = run_representation_experiment(model_factory=cnn_transformer_factory, representations=REPRESENTATIONS, seeds=SEEDS, model_name='CNNTransformer')

In [ ]:
plot_f1_training_curves(cnn_transformer_results, title_prefix='CNNTransformer')

In [ ]:
summary_cnn_transformer = build_summary_df({
    'CNNTransformer': cnn_transformer_results
})

In [ ]:
plot_summary_table(summary_cnn_transformer)

In [ ]:
plot_metrics_comparison(summary_cnn_transformer, title='CNNTransformer — Metrics Comparison')

In [ ]:
plot_f1_comparison(cnn_transformer_results, param_name='Representation', param_name_short='repr')

In [ ]:
best_repr = max(cnn_transformer_results, key=lambda r: np.mean([v['test_macro_f1'] for v in cnn_transformer_results[r].values()]))
best_seed = max(cnn_transformer_results[best_repr], key=lambda s: cnn_transformer_results[best_repr][s]['test_macro_f1'])

best_result = cnn_transformer_results[best_repr][best_seed]
plot_confusion_matrix(best_result['test_cm'])